# Modelling: first model to predict the number of rings

We train two simple regression models and track them with MLflow to compare them.

To open the MLflow UI afterwards, run from the repository root:

```bash
make mlflow-ui
```

## Imports

In [1]:
import mlflow
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

/Users/romaissaemelhaoui/mlops-course/xhec-mlops-2026-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [2]:
df = pd.read_csv("../data/abalone.csv")
df = df[df["Height"] > 0]  # 2 rows with an impossible height of 0 (see EDA)

X = df.drop(columns="Rings")
y = df["Rings"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Preprocessing

`Sex` is one-hot encoded; the numeric measurements are kept as they are.
The encoder and the model are wrapped in a single `Pipeline`, so the same preprocessing is applied at training and prediction time.

In [3]:
def build_pipeline(model) -> Pipeline:
    """Chain the one-hot encoding of `Sex` with a regression model."""
    preprocessor = ColumnTransformer(
        [("sex", OneHotEncoder(handle_unknown="ignore"), ["Sex"])],
        remainder="passthrough",
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])

## Training and tracking with MLflow

In [4]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("abalone-age")

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        pipeline = build_pipeline(model)
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        mlflow.log_params(model.get_params())
        mlflow.log_metrics(
            {
                "rmse": root_mean_squared_error(y_test, y_pred),
                "mae": mean_absolute_error(y_test, y_pred),
                "r2": r2_score(y_test, y_pred),
            }
        )
        mlflow.sklearn.log_model(
            pipeline,
            name="model",
            input_example=X_train.head(),
            # skops (safe format) must be told that the random forest's trees can be loaded
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

2026/10/22 13:03:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/22 13:03:43 INFO mlflow.store.db.utils: Updating database tables


2026/10/22 13:03:44 INFO mlflow.tracking.fluent: Experiment with name 'abalone-age' does not exist. Creating a new experiment.


2026/10/22 13:03:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/22 13:03:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


## Comparing the runs

In [5]:
runs = mlflow.search_runs(experiment_names=["abalone-age"])
runs[["tags.mlflow.runName", "metrics.rmse", "metrics.mae", "metrics.r2"]].round(3)

,tags.mlflow.runName,metrics.rmse,metrics.mae,metrics.r2
0,random_forest,2.240,1.584,0.56
1,linear_regression,2.216,1.591,0.57


## Conclusions

- Both models predict the number of rings with an average error (MAE) of about 1.6 rings and explain about 57% of the variance (R²).
- The random forest is not better than the linear regression (RMSE 2.24 vs 2.22), even though it is much more complex.
- We keep the linear regression as our first model: it is simple, easy to explain, and enough to build the MLOps pipeline in the next steps.